In [28]:
export MODEL="unsloth/Qwen3.6-35B-A3B-GGUF:UD-IQ3_S"
# to set sudo sysctl iogpu.wired_limit_mb=14800. Needs to be set on termimal outside for pure GPU
sysctl iogpu.wired_limit_mb # to check

iogpu.wired_limit_mb: 15000


In [29]:
llama-fit-params -hf $MODEL # This is just for downloading and some initial values to probe

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.009 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [35]:
BATCH_SIZES="2048"  
UBATCH_SIZES="64 128 256 512"
  
echo "Testing different batch/ubatch combinations..."  
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do  
        # Get fitted parameters  
        #OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub | grep -o '\-c [0-9]* \-ngl [0-9]*')  

        OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub 2>&1)
        
        #echo "Raw output: $OUTPUT"  # Debug line  
        
        # Extract context and GPU layers more robustly  
        CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
        NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
        
        if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
            echo "b: $b, ub: $ub, ctx: $CTX, ngl: $NGL"  
        else  
            echo "No valid parameters found"  
        fi  
    done  
done  

Testing different batch/ubatch combinations...
b: 2048, ub: 64, ctx: 58624, ngl: -1
b: 2048, ub: 128, ctx: 55040, ngl: -1
b: 2048, ub: 256, ctx: 48384, ngl: -1
b: 2048, ub: 512, ctx: 35328, ngl: -1


In [34]:
# Pure GPU
llama-bench -hf $MODEL -ngl 99 -fa 1 -t 8 -ub 64,128,256,512 -p 1000,2000 -n 256,512

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.022 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [ ]:
# batched bench seems the closer to llama-server as judged from the logs, thats' why it is here
# llama-batched-bench -hf $MODEL \
#     -ngl 99 --threads 8 -fa on \
#     -c 16000 \
#     -npp 1000 -ntg 1000 -npl 1

In [23]:
llama-fit-params -hf $MODEL -ub 128 # fitt leaves that around of memory in MiB in the target devices

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.009 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [ ]:
llama-server -hf $MODEL \
    --threads-http 1 --threads 8 --no-mmproj --reasoning off \
    --no-warmup -ngl -1 -np 1 --host 0.0.0.0 # -np 1 parameter is important

In [18]:
llama-cli -lv 3 -hf $MODEL --threads 8 --no-mmproj --no-warmup --reasoning off \
    -ngl -1 -st -p "Who are you"

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.022 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet